In [ ]:
"""edit-plan-generation-test.py

Refactored for stable Qwen3-VL video understanding and edit-plan generation.
Architecture:
1) Deterministic metadata extraction (ffprobe)
2) Pass 1 VLM: temporal event extraction (mm:ss.ff)
3) Python validation + timestamp normalization
4) Pass 2 VLM: edit decisions from validated events
5) Deterministic final plan assembly (gapless segments)
"""

'edit-plan-generation-test.py\n\nRefactored for stable Qwen3-VL video understanding and edit-plan generation.\nArchitecture:\n1) Deterministic metadata extraction (ffprobe)\n2) Pass 1 VLM: temporal event extraction (mm:ss.ff)\n3) Python validation + timestamp normalization\n4) Pass 2 VLM: edit decisions from validated events\n5) Deterministic final plan assembly (gapless segments)\n'

In [ ]:
# Cell 1 - Environment Setup

!nvidia-smi
!pip install -U pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install git+https://github.com/huggingface/transformers
!pip install -U accelerate sentencepiece
!pip install -U "qwen-vl-utils[decord]"
# !pip install -U flash-attn --no-build-isolation
# flash attention 2 wheels take too long to build
!apt-get install -y ffmpeg
!pip install -U decord torchcodec

Sun Apr 26 22:31:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   41C    P8             13W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Cell 2 - Mount Drive and Locate Video

from google.colab import drive, files
import os
import re
import json
import shutil
import subprocess
import torch

drive.mount('/content/drive')
video_dir = "/content/drive/MyDrive/test_videos"
os.makedirs(video_dir, exist_ok=True)

existing = os.listdir(video_dir)
if existing:
    print("Videos already on Drive:", existing)
    video_path = os.path.join(video_dir, existing[0])
else:
    uploaded = files.upload()
    filename = next(iter(uploaded.keys()))
    video_path = os.path.join(video_dir, filename)
    shutil.copy(filename, video_path)
    print("Copied to Drive for next time:", video_path)

print("Using video:", video_path)

Mounted at /content/drive
Videos already on Drive: ['pranav-yapping-test-vid.MOV']
Using video: /content/drive/MyDrive/test_videos/pranav-yapping-test-vid.MOV


In [ ]:
# Cell 3 - Hugging Face Login (safe)
# Recommended: store HF_TOKEN in Colab Secrets.

from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(hf_token)
        print("Logged in with Colab secret HF_TOKEN")
    else:
        print("HF_TOKEN secret not found. Continuing without login.")
except Exception as e:
    print(f"Could not load HF_TOKEN secret ({e}). Continuing without login.")


In [ ]:
# Cell 4 - Load Model (Instruct for stable JSON)

from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from qwen_vl_utils import process_vision_info

cache_dir = "/content/drive/MyDrive/hf_models"
os.makedirs(cache_dir, exist_ok=True)

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

# Use Instruct for production extraction.
# Use Thinking only if you explicitly want chain-of-thought style diagnostics.
model_name = "Qwen/Qwen3-VL-8B-Instruct"

model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    attn_implementation="sdpa",
    cache_dir=cache_dir,
).eval()

processor = AutoProcessor.from_pretrained(
    model_name,
    cache_dir=cache_dir,
)

print("Loaded model:", model_name)

Torch: 2.10.0+cu128
CUDA: 12.8
GPU: NVIDIA L4


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

Loaded model: Qwen/Qwen3-VL-8B-Instruct


In [ ]:
# Cell 5 - Helpers

from typing import Any

def run_prompt(
    text_prompt: str,
    video_path: str | None = None,
    sample_fps: float = 2.0,
    max_frames: int = 160,
    total_pixels: int = 8192 * 32 * 32,
    min_pixels: int = 64 * 32 * 32,
    max_new_tokens: int = 4096,
    do_sample: bool = False,
):
    """Run a text prompt with optional video context."""
    content = []

    if video_path:
        content.append(
            {
                "type": "video",
                "video": video_path,
                "total_pixels": total_pixels,
                "min_pixels": min_pixels,
                "max_frames": max_frames,
                "sample_fps": sample_fps,
            }
        )

    content.append({"type": "text", "text": text_prompt})
    conversation = [{"role": "user", "content": content}]

    chat_text = processor.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=True,
    )

    image_inputs, video_inputs, video_kwargs = process_vision_info(
        [conversation],
        return_video_kwargs=True,
        image_patch_size=16,
        return_video_metadata=True,
    )

    if video_inputs is not None:
        video_inputs, video_metadatas = zip(*video_inputs)
        video_inputs = list(video_inputs)
        video_metadatas = list(video_metadatas)
    else:
        video_metadatas = None

    inputs = processor(
        text=[chat_text],
        images=image_inputs,
        videos=video_inputs,
        video_metadata=video_metadatas,
        **video_kwargs,
        do_resize=False,
        return_tensors="pt",
        padding=True,
    ).to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
        )

    trimmed_ids = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    raw_output = processor.batch_decode(
        trimmed_ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    # Instruct usually does not emit <think>, but keep this for compatibility.
    m = re.match(r"<think>\n?(.*?)</think>\n*", raw_output, flags=re.DOTALL)
    if m:
        raw_output = raw_output[m.end():].strip()

    return raw_output.strip()


def get_video_metadata(path: str):
    cmd = [
        "ffprobe",
        "-v",
        "error",
        "-select_streams",
        "v:0",
        "-show_entries",
        "stream=width,height,r_frame_rate,duration",
        "-of",
        "json",
        path,
    ]
    out = subprocess.check_output(cmd).decode("utf-8")
    data = json.loads(out)
    stream = data["streams"][0]

    num, den = stream["r_frame_rate"].split("/")
    fps = float(num) / float(den) if float(den) != 0 else 0.0
    duration_s = float(stream.get("duration", 0) or 0)

    return {
        "duration_s": round(duration_s, 3),
        "width": int(stream["width"]),
        "height": int(stream["height"]),
        "fps": round(fps, 3),
    }


def extract_json_object(text: str):
    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    start = text.find("{")
    end = text.rfind("}")

    if start >= 0 and end > start:
        return json.loads(text[start : end + 1])

    raise ValueError("No valid JSON object found in model output")


def ts_to_seconds(ts: str) -> float:
    m = re.match(r"^(\d+):(\d{2})\.(\d{2})$", str(ts).strip())
    if not m:
        raise ValueError(f"Bad timestamp format: {ts}")

    mm, ss, cs = map(int, m.groups())
    return mm * 60 + ss + cs / 100.0


def seconds_to_ts(seconds: float) -> str:
    seconds = max(0.0, float(seconds))
    mm = int(seconds // 60)
    rest = seconds - (mm * 60)
    ss = int(rest)
    cs = int(round((rest - ss) * 100))

    if cs == 100:
        ss += 1
        cs = 0

    if ss == 60:
        mm += 1
        ss = 0

    return f"{mm:02d}:{ss:02d}.{cs:02d}"


def clamp(x, lo, hi):
    return max(lo, min(hi, x))


def validate_events(events, duration_s: float, min_len_s: float = 1.0):
    cleaned = []

    for e in events:
        try:
            start = clamp(ts_to_seconds(e["start"]), 0, duration_s)
            end = clamp(ts_to_seconds(e["end"]), 0, duration_s)
        except Exception:
            continue

        if end <= start:
            continue

        if end - start < min_len_s:
            continue

        cleaned.append(
            {
                "start": seconds_to_ts(start),
                "end": seconds_to_ts(end),
                "start_s": round(start, 2),
                "end_s": round(end, 2),
                "description": e.get("description", ""),
                "visible_objects": e.get("visible_objects", []),
                "speech_or_text": e.get("speech_or_text", ""),
                "confidence": e.get("confidence", "medium"),
            }
        )

    cleaned.sort(key=lambda x: x["start_s"])

    deduped = []

    for e in cleaned:
        if deduped and e["start_s"] >= deduped[-1]["start_s"] and e["end_s"] <= deduped[-1]["end_s"]:
            continue
        deduped.append(e)

    return deduped


def normalize_segment_decisions(segment_decisions, duration_s: float):
    """
    Converts model-proposed segments into safe renderer segments.
    This makes the final output gapless and clamps bad timestamps.
    """
    normalized = []

    for item in segment_decisions:
        try:
            start = clamp(float(item["start_s"]), 0, duration_s)
            end = clamp(float(item["end_s"]), 0, duration_s)
        except Exception:
            continue

        if end <= start:
            continue

        action = item.get("action", "keep")
        if action not in {"keep", "cut"}:
            action = "keep"

        seg = {
            "start_s": round(start, 2),
            "end_s": round(end, 2),
            "action": action,
        }

        if action == "cut":
            reason = item.get("cut_reason")
            if reason not in {"silence", "filler", "repetition", "off_topic", "pacing", "other"}:
                reason = "pacing"
            seg["cut_reason"] = reason
        else:
            speed = float(item.get("speed", 1.0) or 1.0)
            seg["speed"] = round(clamp(speed, 0.5, 2.0), 2)

            transition = item.get("transition_in") or {"type": "none"}
            if transition.get("type") not in {
                "none",
                "crossfade",
                "fade_from_black",
                "wipe_left",
                "wipe_right",
                "wipe_up",
            }:
                transition = {"type": "none"}

            if transition.get("type") != "none":
                transition["duration_s"] = round(
                    clamp(float(transition.get("duration_s", 0.5)), 0.1, 2.0),
                    2,
                )

            seg["transition_in"] = transition

        normalized.append(seg)

    normalized.sort(key=lambda s: s["start_s"])

    gapless = []
    cursor = 0.0

    for seg in normalized:
        start = seg["start_s"]
        end = seg["end_s"]

        if end <= cursor:
            continue

        if start > cursor:
            gapless.append(
                {
                    "start_s": round(cursor, 2),
                    "end_s": round(start, 2),
                    "action": "keep",
                    "speed": 1.0,
                    "transition_in": {"type": "none"},
                }
            )

        seg["start_s"] = round(max(start, cursor), 2)
        gapless.append(seg)
        cursor = max(cursor, end)

    if cursor < duration_s:
        gapless.append(
            {
                "start_s": round(cursor, 2),
                "end_s": round(duration_s, 2),
                "action": "keep",
                "speed": 1.0,
                "transition_in": {"type": "none"},
            }
        )

    if not gapless:
        gapless = [
            {
                "start_s": 0.0,
                "end_s": round(duration_s, 2),
                "action": "keep",
                "speed": 1.0,
                "transition_in": {"type": "none"},
            }
        ]

    return gapless


def filter_timed_items(items, duration_s: float):
    cleaned = []

    for item in items or []:
        try:
            start = clamp(float(item["start_s"]), 0, duration_s)
            end = clamp(float(item["end_s"]), 0, duration_s)
        except Exception:
            continue

        if end <= start:
            continue

        out = dict(item)
        out["start_s"] = round(start, 2)
        out["end_s"] = round(end, 2)
        cleaned.append(out)

    return cleaned


EDIT_PLAN_CONTRACT = """
Return JSON only. No markdown. No comments.

Required top-level object:
{
  "segments": [
    {
      "start_s": number,
      "end_s": number,
      "action": "keep" | "cut",
      "cut_reason": "silence" | "filler" | "repetition" | "off_topic" | "pacing" | "other",
      "speed": number,
      "transition_in": {
        "type": "none" | "crossfade" | "fade_from_black" | "wipe_left" | "wipe_right" | "wipe_up",
        "duration_s": number
      }
    }
  ],
  "captions": {
    "enabled": boolean,
    "position": "bottom_center" | "top_center" | "center" | "bottom_left" | "bottom_right",
    "grouping": "word_by_word" | "phrase" | "sentence",
    "words": []
  },
  "zooms": [
    {
      "start_s": number,
      "end_s": number,
      "scale": number,
      "anchor": "face" | "center" | "top_third" | "bottom_third" | "custom",
      "easing": "ease_in_out" | "ease_in" | "ease_out" | "linear" | "spring"
    }
  ],
  "overlays": [
    {
      "start_s": number,
      "end_s": number,
      "image_query": string,
      "position": "fullscreen" | "picture_in_picture" | "left_third" | "right_third" | "top_half" | "bottom_half" | "corner_tr" | "corner_tl" | "corner_br" | "corner_bl",
      "animation": "none" | "fade_in" | "slide_in_right" | "slide_in_left" | "slide_in_up" | "pop" | "scale_up"
    }
  ],
  "text_overlays": [
    {
      "start_s": number,
      "end_s": number,
      "text": string,
      "position": "top_center" | "bottom_center" | "center" | "top_left" | "top_right" | "bottom_left" | "bottom_right",
      "style": "title_card" | "lower_third" | "callout" | "stat" | "label",
      "animation": "none" | "fade_in" | "typewriter" | "slide_in_up" | "pop"
    }
  ],
  "music": {
    "enabled": boolean,
    "mood": "upbeat" | "chill" | "dramatic" | "corporate" | "playful" | "inspirational" | "dark" | "none",
    "start_s": number,
    "end_s": number,
    "volume": number,
    "duck_under_speech": boolean
  },
  "reframe": {
    "enabled": boolean,
    "target_aspect_ratio": "16:9" | "9:16" | "1:1" | "4:5",
    "focus": "face_track" | "center" | "custom"
  }
}
""".strip()


def build_final_plan(model_plan, source_meta):
    final_edit_plan = {
        "source_video": {
            "duration_s": source_meta["duration_s"],
            "width": source_meta["width"],
            "height": source_meta["height"],
            "fps": source_meta["fps"],
        },
        "segments": normalize_segment_decisions(
            model_plan.get("segments", []),
            source_meta["duration_s"],
        ),
        "captions": model_plan.get("captions") or {
            "enabled": True,
            "position": "bottom_center",
            "grouping": "phrase",
            "words": [],
        },
        "zooms": filter_timed_items(model_plan.get("zooms", []), source_meta["duration_s"]),
        "overlays": filter_timed_items(model_plan.get("overlays", []), source_meta["duration_s"]),
        "text_overlays": filter_timed_items(model_plan.get("text_overlays", []), source_meta["duration_s"]),
        "music": model_plan.get("music") or {
            "enabled": False,
            "mood": "none",
            "start_s": 0,
            "end_s": source_meta["duration_s"],
            "volume": 0.15,
            "duck_under_speech": True,
        },
        "reframe": model_plan.get("reframe") or {
            "enabled": False,
            "target_aspect_ratio": "9:16",
            "focus": "face_track",
        },
    }

    # Since Whisper word timings are not provided in this notebook, keep this empty.
    final_edit_plan["captions"]["words"] = []

    final_edit_plan["music"].setdefault("enabled", False)
    final_edit_plan["music"].setdefault("mood", "none")
    final_edit_plan["music"].setdefault("start_s", 0)
    final_edit_plan["music"].setdefault("end_s", source_meta["duration_s"])
    final_edit_plan["music"].setdefault("volume", 0.15)
    final_edit_plan["music"].setdefault("duck_under_speech", True)

    return final_edit_plan

def default_keep_plan(source_meta):
    duration_s = source_meta["duration_s"]

    return {
        "source_video": {
            "duration_s": source_meta["duration_s"],
            "width": source_meta["width"],
            "height": source_meta["height"],
            "fps": source_meta["fps"],
        },
        "segments": [
            {
                "start_s": 0.0,
                "end_s": round(duration_s, 2),
                "action": "keep",
                "speed": 1.0,
                "transition_in": {"type": "none"},
            }
        ],
        "captions": {
            "enabled": True,
            "position": "bottom_center",
            "grouping": "phrase",
            "words": [],
        },
        "zooms": [],
        "overlays": [],
        "text_overlays": [],
        "music": {
            "enabled": False,
            "mood": "none",
            "start_s": 0,
            "end_s": round(duration_s, 2),
            "volume": 0.15,
            "duck_under_speech": True,
        },
        "reframe": {
            "enabled": False,
            "target_aspect_ratio": "9:16",
            "focus": "face_track",
        },
    }


def make_gapless_segments(segments, duration_s, eps=0.05):
    """
    Renderer-safe cleanup:
    - clamps timestamps
    - drops zero-length segments
    - sorts segments
    - fills real gaps with keep segments
    - avoids final zero-length segment caused by rounding
    """
    cleaned = []
    duration_rounded = round(float(duration_s), 2)

    for s in segments or []:
        try:
            start = clamp(float(s["start_s"]), 0, duration_s)
            end = clamp(float(s["end_s"]), 0, duration_s)
        except Exception:
            continue

        # Drop invalid or tiny segments before rounding.
        if end - start <= eps:
            continue

        action = s.get("action", "keep")
        if action not in {"keep", "cut"}:
            action = "keep"

        out = {
            "start_s": round(start, 2),
            "end_s": round(end, 2),
            "action": action,
        }

        # Drop any segment that became zero-length after rounding.
        if out["end_s"] <= out["start_s"]:
            continue

        if action == "cut":
            reason = s.get("cut_reason", "pacing")
            if reason not in {"silence", "filler", "repetition", "off_topic", "pacing", "other"}:
                reason = "pacing"
            out["cut_reason"] = reason
        else:
            try:
                speed = float(s.get("speed", 1.0))
            except Exception:
                speed = 1.0

            out["speed"] = round(clamp(speed, 0.5, 2.0), 2)

            transition = s.get("transition_in") or {"type": "none"}

            # Model sometimes outputs "transition_in": "none"
            # but the renderer schema expects {"type": "none"}.
            if isinstance(transition, str):
                transition = {"type": transition}

            # If model outputs something invalid like a list/number, reset safely.
            if not isinstance(transition, dict):
                transition = {"type": "none"}

            transition_type = transition.get("type", "none")

            if transition_type not in {
                "none",
                "crossfade",
                "fade_from_black",
                "wipe_left",
                "wipe_right",
                "wipe_up",
            }:
                transition_type = "none"

            clean_transition = {"type": transition_type}

            if transition_type != "none":
                try:
                    duration = float(transition.get("duration_s", 0.5))
                except Exception:
                    duration = 0.5

                clean_transition["duration_s"] = round(clamp(duration, 0.1, 2.0), 2)

            out["transition_in"] = clean_transition

        cleaned.append(out)

    cleaned.sort(key=lambda x: x["start_s"])

    gapless = []
    cursor = 0.0

    for s in cleaned:
        start = s["start_s"]
        end = s["end_s"]

        if end <= cursor + eps:
            continue

        # Fill only meaningful gaps.
        if start > cursor + eps:
            filler = {
                "start_s": round(cursor, 2),
                "end_s": round(start, 2),
                "action": "keep",
                "speed": 1.0,
                "transition_in": {"type": "none"},
            }

            if filler["end_s"] > filler["start_s"]:
                gapless.append(filler)

        s["start_s"] = round(max(start, cursor), 2)

        if s["end_s"] > s["start_s"]:
            gapless.append(s)
            cursor = max(cursor, s["end_s"])

    # Final tail: avoid 37.3 -> 37.3 zero-length segment.
    if duration_s - cursor > eps:
        tail = {
            "start_s": round(cursor, 2),
            "end_s": duration_rounded,
            "action": "keep",
            "speed": 1.0,
            "transition_in": {"type": "none"},
        }

        if tail["end_s"] > tail["start_s"]:
            gapless.append(tail)

    if not gapless:
        return [
            {
                "start_s": 0.0,
                "end_s": duration_rounded,
                "action": "keep",
                "speed": 1.0,
                "transition_in": {"type": "none"},
            }
        ]

    # Force exact clean boundaries after rounding.
    gapless[0]["start_s"] = 0.0
    gapless[-1]["end_s"] = duration_rounded

    # Final safety pass: remove any accidental zero-length segments.
    gapless = [s for s in gapless if s["end_s"] > s["start_s"]]

    return gapless

In [ ]:
# Cell 6 - Pass 1: Timeline extraction

source_meta = get_video_metadata(video_path)

print("Source metadata:")
print(json.dumps(source_meta, indent=2))

PROMPT_TIMELINE = f"""
You are analyzing a video for an editing pipeline.

Source metadata:
{json.dumps(source_meta, indent=2)}

Return JSON only with this exact shape:
{{
  "summary": "short summary",
  "events": [
    {{
      "start": "mm:ss.ff",
      "end": "mm:ss.ff",
      "description": "short visual description",
      "visible_objects": ["object"],
      "confidence": "low|medium|high"
    }}
  ]
}}

Rules:
- Output at most 6 events.
- Timestamps must be in mm:ss.ff format.
- Each event must be at least 1 second long.
- Each description must be under 100 characters.
- Do NOT read, transcribe, or guess text on objects.
- Do NOT include product labels, book titles, brands, logos, or long numbers.
- Do NOT include markdown fences.
- If unsure, say "person holding object" or "person speaking".
""".strip()

response_1 = run_prompt(
    PROMPT_TIMELINE,
    video_path=video_path,
    sample_fps=2,
    max_frames=120,
    total_pixels=8192 * 32 * 32,
    min_pixels=64 * 32 * 32,
    max_new_tokens=1200,
    do_sample=False,
)

print("=" * 60)
print("PASS 1 RESPONSE")
print("=" * 60)
print(response_1)

try:
    timeline_json = extract_json_object(response_1)
    raw_events = timeline_json.get("events", [])
    clean_events = validate_events(raw_events, source_meta["duration_s"], min_len_s=1.0)

    if not clean_events:
        raise ValueError("No valid events found.")

except Exception as e:
    print("Timeline extraction failed. Falling back to one full-video event.")
    print(e)

    timeline_json = {
        "summary": "Fallback timeline.",
        "events": []
    }

    raw_events = []
    clean_events = [
        {
            "start": "00:00.00",
            "end": seconds_to_ts(source_meta["duration_s"]),
            "start_s": 0.0,
            "end_s": round(source_meta["duration_s"], 2),
            "description": "Full video",
            "visible_objects": [],
            "confidence": "low",
        }
    ]

print(f"Raw events: {len(raw_events)}")
print(f"Validated events: {len(clean_events)}")
print(json.dumps(clean_events[:8], indent=2))

Source metadata:
{
  "duration_s": 37.302,
  "width": 1920,
  "height": 1080,
  "fps": 30.0
}


qwen-vl-utils using torchcodec to read video.
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.10.0+cu128) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call last):
  Fi

PASS 1 RESPONSE
{
  "summary": "Person shows various items including a book, box, and bottle.",
  "events": [
    {
      "start": "00:00.00",
      "end": "00:04.00",
      "description": "Person waves hand while standing in a room.",
      "visible_objects": ["person"],
      "confidence": "high"
    },
    {
      "start": "00:04.00",
      "end": "00:16.00",
      "description": "Person holds and flips through a book.",
      "visible_objects": ["person", "book"],
      "confidence": "high"
    },
    {
      "start": "00:16.00",
      "end": "00:22.00",
      "description": "Person holds up a blue box and turns it around.",
      "visible_objects": ["person", "box"],
      "confidence": "high"
    },
    {
      "start": "00:22.00",
      "end": "00:37.00",
      "description": "Person holds and rotates a small bottle.",
      "visible_objects": ["person", "bottle"],
      "confidence": "high"
    }
  ]
}
Raw events: 4
Validated events: 4
[
  {
    "start": "00:00.00",
    "end": 

In [ ]:
# Cell 7 - Pass 2: Renderer-ready edit plan generation

PROMPT_RENDERER_PLAN = f"""
You are the editor for a deterministic video renderer.

You must make real editorial decisions based on the visual timeline and transcript.
Do not simply keep everything unless the whole video is genuinely useful.

Source metadata:
{json.dumps(source_meta, indent=2)}

Visual timeline:
{json.dumps(clean_events, indent=2)}

Transcript with word timings:
{json.dumps(whisper_words, indent=2)}

Return one JSON object matching this contract:
- segments: must cover 0.0 to {source_meta["duration_s"]}, no gaps, no overlaps.
- captions: choose enabled, position, grouping, and add emphasis to important words.
- zooms: add only on emphasis moments, usually 1.15-1.35 scale.
- overlays: add only when a visual reference would help.
- text_overlays: add short labels for key takeaways.
- music: choose whether music helps; disable it if it distracts.
- reframe: choose 9:16 face_track if this is intended for Shorts/Reels/TikTok.

Rules:
- Output JSON only.
- Do not include markdown.
- Use only schema-valid enum values.
- Do not invent transcript words.
- Every cut must have a cut_reason.
- Avoid random edits. Every zoom, overlay, or text overlay must match the content.
""".strip()

response_2 = run_prompt(
    PROMPT_RENDERER_PLAN,
    video_path=video_path,
    sample_fps=1,
    max_frames=160,
    total_pixels=20480 * 32 * 32,
    min_pixels=64 * 32 * 32,
    max_new_tokens=5000,
    do_sample=False,
)

print("=" * 60)
print("PASS 2 RESPONSE")
print("=" * 60)
print(response_2)

try:
    model_plan = extract_json_object(response_2)
except Exception as e:
    print("Edit plan generation failed. Falling back to keep-only plan.")
    print(e)
    model_plan = default_keep_plan(source_meta)

NameError: name 'whisper_words' is not defined

In [ ]:
# Cell 8 - Final assembly + print only

# If model_plan already has source_video, ignore it and use deterministic ffprobe metadata.
final_edit_plan = {
    "source_video": {
        "duration_s": source_meta["duration_s"],
        "width": source_meta["width"],
        "height": source_meta["height"],
        "fps": source_meta["fps"],
    },
    "segments": make_gapless_segments(
        model_plan.get("segments", []),
        source_meta["duration_s"]
    ),
    "captions": model_plan.get("captions", {
        "enabled": True,
        "position": "bottom_center",
        "grouping": "phrase",
        "words": [],
    }),
    "zooms": model_plan.get("zooms", []),
    "overlays": model_plan.get("overlays", []),
    "text_overlays": model_plan.get("text_overlays", []),
    "music": model_plan.get("music", {
        "enabled": False,
        "mood": "none",
        "start_s": 0,
        "end_s": source_meta["duration_s"],
        "volume": 0.15,
        "duck_under_speech": True,
    }),
    "reframe": model_plan.get("reframe", {
        "enabled": False,
        "target_aspect_ratio": "9:16",
        "focus": "face_track",
    }),
}

# Force captions.words empty because Whisper word timings are not present.
final_edit_plan["captions"]["words"] = []

segments = final_edit_plan["segments"]

print("Segments:", len(segments))
print("Coverage:", {
    "start": segments[0]["start_s"] if segments else None,
    "end": segments[-1]["end_s"] if segments else None,
    "duration": source_meta["duration_s"],
})

print("=" * 60)
print("FINAL EDIT PLAN JSON")
print("=" * 60)
print(json.dumps(final_edit_plan, indent=2))

In [ ]:
# Cell 7 - Pass 2: Visual-only renderer-ready edit plan generation

PHASE = "visual_only"

PROMPT_RENDERER_PLAN = f"""
You are generating an edit plan for a deterministic video renderer.

This is PHASE 1: visual-only editing.
No Whisper transcript is available.
No word-level caption timings are available.
You must make the best possible edit decisions from visual content only.

Source metadata:
{json.dumps(source_meta, indent=2)}

Validated visual timeline from Pass 1:
{json.dumps(clean_events, indent=2)}

Your job:
Create a JSON edit plan that fits the renderer schema.
The renderer will execute your plan exactly.
Do not rely on the renderer to make creative decisions.

Important:
- You are allowed to keep the whole video if the visuals are continuous and useful.
- But do not simply output empty/default edits if a visual edit would improve pacing.
- Use zooms when the speaker/object/action visually becomes important.
- Use text_overlays only for visual section labels or visual emphasis.
- Use overlays only when the visible content strongly suggests a helpful inserted image.
- Since there is no transcript, do not invent spoken words.
- Since there are no Whisper timings, captions.words must be [].
- In visual-only mode, captions.enabled should usually be false.

Return JSON only with these top-level keys:
{{
  "segments": [],
  "captions": {{}},
  "zooms": [],
  "overlays": [],
  "text_overlays": [],
  "music": {{}},
  "reframe": {{}}
}}

Field rules:

segments:
- Must cover the whole source video from 0.0 to {source_meta["duration_s"]}.
- No gaps.
- No overlaps.
- Use action "keep" or "cut".
- Use "cut" only for visually obvious dead time, setup time, awkward pauses, repeated actions, or pacing issues.
- Every cut segment must include cut_reason.
- Every keep segment should include speed and transition_in.
- Keep speed between 0.5 and 2.0. Usually use 1.0.

captions:
- Must include enabled, position, grouping, words.
- words must be [].
- Because no transcript exists, enabled should usually be false.

zooms:
- Add 0 to 4 zooms if useful.
- Good zoom moments: object reveal, facial reaction, gesture, important visual action.
- Use scale around 1.15 to 1.35 for normal punch-ins.
- Use anchor "face" for talking-head moments, "center" for object/action moments.
- Do not zoom randomly.

overlays:
- Add only if a visual overlay would clearly help the edit.
- image_query must describe what image the pipeline should search for.
- Do not add overlays based on guessezooms:
- Add 0 to 4 zooms if useful.
- Good zoom moments: object reveal, facial reaction, gesture, important visual action.
- Use scale around 1.15 to 1.35 for normal punch-ins.
- Use anchor "face" for talking-head moments, "center" for object/action moments.
- Do not zoom randomly.

overlays:
- Add only if a visual overlay would clearly help the edit.
- image_query must describe what image the pipeline should search for.
- Do not add overlays based on guessedd speech/audio.
- Prefer [] if visuals alone do not justify overlays.

text_overlays:
- Add 0 to 3 short visual labels if useful.
- Text must be under 80 characters.
- Do not pretend to know the exact spoken message.
- Good examples: "Object reveal", "Key visual moment", "Before", "After".
- Prefer [] if labels would feel generic.

music:
- Choose enabled true only if background music would likely improve the edit.
- If enabled is false, use mood "none".
- Keep volume between 0.1 and 0.2 if enabled.

reframe:
- If the video looks like a talking-head or social clip, consider enabled true with target_aspect_ratio "9:16" and focus "face_track".
- If the original framing is important, use enabled false.

Valid enum values:

cut_reason:
"silence", "filler", "repetition", "off_topic", "pacing", "other"

transition_in.type:
"none", "crossfade", "fade_from_black", "wipe_left", "wipe_right", "wipe_up"

captions.position:
"bottom_center", "top_center", "center", "bottom_left", "bottom_right"

captions.grouping:
"word_by_word", "phrase", "sentence"

zoom.anchor:
"face", "center", "top_third", "bottom_third", "custom"

zoom.easing:
"ease_in_out", "ease_in", "ease_out", "linear", "spring"

overlay.position:
"fullscreen", "picture_in_picture", "left_third", "right_third", "top_half", "bottom_half", "corner_tr", "corner_tl", "corner_br", "corner_bl"

overlay.animation:
"none", "fade_in", "slide_in_right", "slide_in_left", "slide_in_up", "pop", "scale_up"

text_overlay.position:
"top_center", "bottom_center", "center", "top_left", "top_right", "bottom_left", "bottom_right"

text_overlay.style:
"title_card", "lower_third", "callout", "stat", "label"

text_overlay.animation:
"none", "fade_in", "typewriter", "slide_in_up", "pop"

music.mood:
"upbeat", "chill", "dramatic", "corporate", "playful", "inspirational", "dark", "none"

reframe.target_aspect_ratio:
"16:9", "9:16", "1:1", "4:5"

reframe.focus:
"face_track", "center", "custom"

Output rules:
- Return JSON only.
- No markdown fences.
- No explanation.
- Do not include source_video. Python will add source_video deterministically.
- Do not include fields outside the schema.
""".strip()

response_2 = run_prompt(
    PROMPT_RENDERER_PLAN,
    video_path=video_path,          # important: give Pass 2 the video too
    sample_fps=2,
    max_frames=120,
    total_pixels=8192 * 32 * 32,
    min_pixels=64 * 32 * 32,
    max_new_tokens=4000,
    do_sample=False,
)

print("=" * 60)
print("PASS 2 RESPONSE")
print("=" * 60)
print(response_2)

try:
    model_plan = extract_json_object(response_2)
except Exception as e:
    print("Edit plan generation failed. Falling back to keep-only plan.")
    print(e)
    model_plan = default_keep_plan(source_meta)

          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.10.0+cu128) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/to

PASS 2 RESPONSE
{
  "segments": [
    {
      "start_s": 0.0,
      "end_s": 4.0,
      "action": "keep",
      "speed": 1.0,
      "transition_in": "none"
    },
    {
      "start_s": 4.0,
      "end_s": 16.0,
      "action": "keep",
      "speed": 1.0,
      "transition_in": "none"
    },
    {
      "start_s": 16.0,
      "end_s": 22.0,
      "action": "keep",
      "speed": 1.0,
      "transition_in": "none"
    },
    {
      "start_s": 22.0,
      "end_s": 37.0,
      "action": "keep",
      "speed": 1.0,
      "transition_in": "none"
    }
  ],
  "captions": {
    "enabled": false,
    "position": "bottom_center",
    "grouping": "sentence",
    "words": []
  },
  "zooms": [
    {
      "start_s": 32.0,
      "end_s": 35.0,
      "scale": 1.25,
      "anchor": "center",
      "easing": "ease_in_out"
    }
  ],
  "overlays": [],
  "text_overlays": [],
  "music": {
    "enabled": false,
    "mood": "none"
  },
  "reframe": {
    "enabled": false
  }
}


In [ ]:
# Cell 8 - Final assembly + print only

final_edit_plan = {
    "source_video": {
        "duration_s": source_meta["duration_s"],
        "width": source_meta["width"],
        "height": source_meta["height"],
        "fps": source_meta["fps"],
    },
    "segments": make_gapless_segments(
        model_plan.get("segments", []),
        source_meta["duration_s"]
    ),
    "captions": model_plan.get("captions") or {
        "enabled": False,
        "position": "bottom_center",
        "grouping": "phrase",
        "words": [],
    },
    "zooms": filter_timed_items(
        model_plan.get("zooms", []),
        source_meta["duration_s"]
    ),
    "overlays": filter_timed_items(
        model_plan.get("overlays", []),
        source_meta["duration_s"]
    ),
    "text_overlays": filter_timed_items(
        model_plan.get("text_overlays", []),
        source_meta["duration_s"]
    ),
    "music": model_plan.get("music") or {
        "enabled": False,
        "mood": "none",
        "start_s": 0,
        "end_s": source_meta["duration_s"],
        "volume": 0.15,
        "duck_under_speech": True,
    },
    "reframe": model_plan.get("reframe") or {
        "enabled": False,
        "target_aspect_ratio": "9:16",
        "focus": "face_track",
    },
}

# Phase 1: visual-only. No Whisper timings exist yet.
# Keep caption object schema-valid, but force no word captions.
final_edit_plan["captions"]["words"] = []

if "enabled" not in final_edit_plan["captions"]:
    final_edit_plan["captions"]["enabled"] = False

if "position" not in final_edit_plan["captions"]:
    final_edit_plan["captions"]["position"] = "bottom_center"

if "grouping" not in final_edit_plan["captions"]:
    final_edit_plan["captions"]["grouping"] = "phrase"

# Since words are empty, disable captions for Phase 1.
final_edit_plan["captions"]["enabled"] = False

# Music defaults / cleanup.
final_edit_plan["music"].setdefault("enabled", False)
final_edit_plan["music"].setdefault("mood", "none")
final_edit_plan["music"].setdefault("start_s", 0)
final_edit_plan["music"].setdefault("end_s", source_meta["duration_s"])
final_edit_plan["music"].setdefault("volume", 0.15)
final_edit_plan["music"].setdefault("duck_under_speech", True)

if not final_edit_plan["music"]["enabled"]:
    final_edit_plan["music"]["mood"] = "none"

# Reframe defaults / cleanup.
final_edit_plan["reframe"].setdefault("enabled", False)
final_edit_plan["reframe"].setdefault("target_aspect_ratio", "9:16")
final_edit_plan["reframe"].setdefault("focus", "face_track")

# Final sanity check for segments.
segments = final_edit_plan["segments"]
segments = [s for s in segments if s["end_s"] > s["start_s"]]
final_edit_plan["segments"] = segments

print("Segments:", len(segments))
print("Coverage:", {
    "start": segments[0]["start_s"] if segments else None,
    "end": segments[-1]["end_s"] if segments else None,
    "duration": source_meta["duration_s"],
})

print("=" * 60)
print("FINAL EDIT PLAN JSON")
print("=" * 60)
print(json.dumps(final_edit_plan, indent=2))

Segments: 5
Coverage: {'start': 0.0, 'end': 37.3, 'duration': 37.302}
FINAL EDIT PLAN JSON
{
  "source_video": {
    "duration_s": 37.302,
    "width": 1920,
    "height": 1080,
    "fps": 30.0
  },
  "segments": [
    {
      "start_s": 0.0,
      "end_s": 4.0,
      "action": "keep",
      "speed": 1.0,
      "transition_in": {
        "type": "none"
      }
    },
    {
      "start_s": 4.0,
      "end_s": 16.0,
      "action": "keep",
      "speed": 1.0,
      "transition_in": {
        "type": "none"
      }
    },
    {
      "start_s": 16.0,
      "end_s": 22.0,
      "action": "keep",
      "speed": 1.0,
      "transition_in": {
        "type": "none"
      }
    },
    {
      "start_s": 22.0,
      "end_s": 37.0,
      "action": "keep",
      "speed": 1.0,
      "transition_in": {
        "type": "none"
      }
    },
    {
      "start_s": 37.0,
      "end_s": 37.3,
      "action": "keep",
      "speed": 1.0,
      "transition_in": {
        "type": "none"
      }
    }
  ],